# Merge LoRA + quantize to GGUF (for CPU serving)

Turns your trained `lora_adapter` into a single **quantized GGUF** (~2.5 GB) that runs on **CPU** via llama.cpp — no GPU needed. Drop the result into `artifacts/rationale-model/` and the app's `LLM_MODE=local` will use it.

Run on Colab (any GPU, or even CPU — GPU just loads the 4B base faster). Upload your `lora_adapter.zip` when prompted.

## 1. Install

In [ ]:
!pip -q install -U transformers peft accelerate sentencepiece
import torch; print('cuda:', torch.cuda.is_available())

## 2. Upload + unzip the adapter

In [ ]:
from google.colab import files
up = files.upload()             # choose lora_adapter.zip
import zipfile, os
zname = next(iter(up))
with zipfile.ZipFile(zname) as z: z.extractall('.')
ADAPTER = 'lora_adapter' if os.path.isdir('lora_adapter') else zname.replace('.zip','')
print('adapter dir:', ADAPTER, os.listdir(ADAPTER))

## 3. Merge LoRA into the base model

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = json.load(open(f'{ADAPTER}/adapter_config.json'))['base_model_name_or_path']
print('base model:', BASE)
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained('merged_model', safe_serialization=True)
tok.save_pretrained('merged_model')
print('merged ->', os.listdir('merged_model'))

## 4. Convert to GGUF (f16) with llama.cpp

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py merged_model --outfile qwen-rationale-f16.gguf --outtype f16

## 5. Build llama.cpp + quantize to Q4_K_M (~2.5 GB)

In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null 2>&1
!cmake --build llama.cpp/build --config Release -j --target llama-quantize > /dev/null 2>&1
!./llama.cpp/build/bin/llama-quantize qwen-rationale-f16.gguf qwen-rationale-q4.gguf Q4_K_M
!ls -lh qwen-rationale-q4.gguf

## 6. Download the quantized GGUF

In [ ]:
from google.colab import files
files.download('qwen-rationale-q4.gguf')

---
**On your machine:** put the file at `artifacts/rationale-model/qwen-rationale-q4.gguf`, then:
```powershell
pip install llama-cpp-python
$env:LLM_MODE = "local"
```
Now the app generates rationales with your fine-tuned model on CPU. See `docs/instructions/07_usage.md`.